# 07 · pass@k sampling

The standard **pass@k** evaluation metric and how to compute
it efficiently with concurrent sandboxes.

**Key concepts**
- **pass@k** — sample *k* candidate solutions (temperature > 0); the task is solved
  if *any* passes. `pass@1` reflects single-shot quality; `pass@k` reflects how much
  sampling helps.
- **Concurrency** — the k candidates are generated and evaluated in parallel, each
  in its own `AsyncSandbox`, so k-way sampling doesn't cost k× wall-clock.
- **Isolation** — separate sandboxes keep candidate runs independent.

**How it works:** sample k solutions, run each against the hidden tests concurrently,
then report passes/k, pass@1, and pass@k.

**Prerequisites:** AWS credentials + Bedrock model access. `PASS_AT_K` sets k.

In [ ]:
%pip install -q raitbox

### Configuration & sampler
Bedrock client, the task + hidden tests, and `sample_solution` — called at temperature > 0 so samples differ.

In [ ]:
import asyncio
import os
import re

import boto3

from raitbox import AsyncSandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
K = int(os.environ.get("PASS_AT_K", "5"))

SYSTEM_PROMPT = [{"text": "Implement exactly the requested function. Reply with ONLY "
                          "the function definition — no prose, no fences, no tests."}]

TASK = {
    "prompt": "Implement `def longest_common_prefix(strs: list[str]) -> str` returning "
              "the longest common prefix of the list ('' if none).",
    "tests": "assert longest_common_prefix(['flower','flow','flight']) == 'fl'; "
             "assert longest_common_prefix(['dog','car']) == ''; "
             "assert longest_common_prefix(['abc']) == 'abc'",
}


def sample_solution(temperature: float) -> str:
    resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM_PROMPT,
                            messages=[{"role": "user", "content": [{"text": TASK["prompt"]}]}],
                            inferenceConfig={"maxTokens": 800, "temperature": temperature})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()

### Concurrent evaluation
Each candidate runs in its own `AsyncSandbox`; `pass_at_k` samples k solutions and evaluates them in parallel.

In [ ]:
async def evaluate_one(code: str) -> bool:
    program = code + "\n" + TASK["tests"] + "\nprint('OK')"
    async with await AsyncSandbox.create(region_name=REGION) as sbx:
        ex = await sbx.run_code(program)
    return ex.error is None and "OK" in "".join(ex.logs.stdout)


async def pass_at_k(k: int):
    candidates = await asyncio.gather(
        *(asyncio.to_thread(sample_solution, 0.8) for _ in range(k))
    )
    verdicts = await asyncio.gather(*(evaluate_one(c) for c in candidates))
    return candidates, verdicts

## Run it

In [ ]:
candidates, verdicts = await pass_at_k(K)

passes = sum(verdicts)
print(f"k = {K}")
for i, v in enumerate(verdicts):
    print(f"  sample {i}: {'PASS' if v else 'fail'}")
print(f"\npasses: {passes}/{K}")
print(f"pass@1 (first sample): {verdicts[0]}")
print(f"pass@{K} (any correct): {passes > 0}")

## Recap & where to go next
You computed a real sampling-based eval metric using concurrent isolated sandboxes.

**Related:** `01-basics` (async basics) · `05` (single-sample scoring) · `10` (parallel
eval over a dataset).